In [ ]:
!pip install fasttext transformers mlflow spacy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 4.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 6.9 MB/s eta 0:00:00
  Using cached pybind11-3.0.4-py3-none-any.whl.metadata (10 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 219.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 175.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 151.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 251.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 22.4 MB/s eta 0:00:00
   

In [ ]:
!python -m spacy download ru_core_news_md

In [89]:
import os
import random
import sys

import fasttext
import fasttext.util
import mlflow.transformers
import numpy as np
import polars as pl
import torch
from sklearn.metrics import classification_report
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive

    drive.mount('gdrive')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = (
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)
DEVICE

Drive already mounted at gdrive; to attempt to forcibly remount, call drive.mount("gdrive", force_remount=True).


'cuda'

In [ ]:
from abc import ABC, abstractmethod

class TextTokenizer(ABC):

    @abstractmethod
    def encode(self, texts: list[str]) -> tuple[torch.Tensor, torch.Tensor]:
        pass

    @abstractmethod
    def decode(self, id: int) -> str:
        pass

class ToxicityClassifier(ABC, torch.nn.Module):

    @property
    def device(self):
        return next(self.parameters()).device

    @abstractmethod
    def log_model(self, tokenizer: TextTokenizer, registered_model_name: str):
        pass

class CommentsDataset(torch.utils.data.Dataset):
    """
    Torch dataset for loading comments from a csv file
    """
    LABEL_MAPPING = {
        "NORMAL": 0,
        "INSULT": 1,
        "THREAT": 2,
        "OBSCENITY": 3
    }
    def __init__(self, filename: str):
        data = (
            pl.scan_csv(filename)
            .with_columns(
                pl.col("label")
                .replace_strict(self.LABEL_MAPPING)
                .alias("label")
            ).collect()
        )
        self.comments = data["comment"]
        self.labels = data["label"]

    def __len__(self):
        return len(self.comments)

    def __getitem__(self, idx):
        return self.comments[idx], self.labels[idx]

    @property
    def lengths(self):
        return self.comments.str.len_chars().to_list()

    @property
    def label_idx(self):
        return sorted(self.LABEL_MAPPING.values())

    @property
    def label_names(self):
        reverse = { value: key for key, value in self.LABEL_MAPPING.items() }
        return [reverse[id] for id in self.label_idx]



class BucketBatchSampler(torch.utils.data.Sampler):
    def __init__(self, lengths, batch_size, bucket_size=1000, shuffle=True):
        self.lengths = lengths
        self.batch_size = batch_size
        self.bucket_size = bucket_size
        self.shuffle = shuffle

    def __iter__(self):
        indices = list(range(len(self.lengths)))

        if self.shuffle:
            random.shuffle(indices)

        buckets = [
            indices[i:i + self.bucket_size]
            for i in range(0, len(indices), self.bucket_size)
        ]

        batches = []

        for bucket in buckets:
            bucket.sort(key=lambda i: self.lengths[i])

            for i in range(0, len(bucket), self.batch_size):
                batch = bucket[i:i + self.batch_size]
                batches.append(batch)

        if self.shuffle:
            random.shuffle(batches)

        return iter(batches)

    def __len__(self):
        return (len(self.lengths) + self.batch_size - 1) // self.batch_size


def pad_collate(batch: list[tuple[str, int]], tokenizer: TextTokenizer):
    """
    Pad to maximum length in a batch
    :returns Tuple with values:
    * tensor (batch_size, max_seq_len) -
    * tensor (batch_size, ) - real length of text sequencies
    * tensor (batch_size, ) - labels
    """
    comments, labels = zip(*batch)
    batch_tokenized, lengths = tokenizer.encode(comments)
    return batch_tokenized, lengths, torch.tensor(labels)

In [ ]:
DATA_DIRECTORY = "gdrive/MyDrive/data/" if IS_COLAB else "../data/interim/"
MLFLOW_TRACKING_URI = "http://70.34.242.179"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
os.environ["MLFLOW_LOG_MODEL_COMPRESSION"] = "gzip"

In [ ]:
def get_optimizer_params(optimizer, prefix="optimizer"):
    for i, group in enumerate(optimizer.param_groups):
        params = {
            f"{prefix}.group_{i}.lr": group.get("lr"),
            f"{prefix}.group_{i}.weight_decay": group.get("weight_decay"),
            f"{prefix}.group_{i}.momentum": group.get("momentum"),
            f"{prefix}.group_{i}.betas": str(group.get("betas")),
            f"{prefix}.group_{i}.eps": group.get("eps"),
        }
        params = {k: v for k, v in params.items() if v is not None}
        return params

def get_scheduler_params(scheduler, prefix="scheduler"):
    state = scheduler.state_dict()
    params = {}
    for key, value in state.items():
        if isinstance(value, (int, float, str, bool)):
            params[f"{prefix}.{key}"] = value
        else:
            params[f"{prefix}.{key}"] = str(value)
    return params

def train(model, optimizer, criterion, train_loader, device, pbar):
    train_cum_loss = 0
    model.train()
    for X, lengths, labels in train_loader:
        X = X.to(device=device)
        labels = labels.to(device=device)
        lengths = lengths.to(device=device)

        optimizer.zero_grad()

        pred = model(X, lengths)
        loss = criterion(pred, labels)
        loss.backward()

        optimizer.step()

        train_cum_loss += loss.item() * X.size(0)
        pbar.update()

    return train_cum_loss / len(train_loader)

def validate(model, optimizer, criterion, val_loader, device, pbar):
    class_labels = val_loader.dataset.label_names
    class_ids = val_loader.dataset.label_idx

    model.eval()
    val_cum_loss = 0
    all_predicted_probas = []
    all_true_labels = []
    with torch.inference_mode():
        for texts, lengths, labels in val_loader:
            texts = texts.to(device=device)
            lengths = lengths.to(device=device)
            labels = labels.to(device=device)
            logits = model(texts, lengths)
            val_cum_loss += criterion(logits, labels).item() * labels.size(0)
            pred_probas = torch.nn.functional.softmax(logits, dim=1).detach()
            all_predicted_probas.extend(pred_probas.cpu().tolist())
            all_true_labels.extend(labels.cpu().tolist())

    report = classification_report(
        all_true_labels,
        np.array(all_predicted_probas).argmax(axis=1),
        labels=class_ids,
        target_names=class_labels,
        output_dict=True,
        zero_division=0.0
    )

    return val_cum_loss / len(val_loader), report

def build_loader(data_file, lenght_based_padding, weighted_sampler, batch_size):
    dataset = CommentsDataset(DATA_DIRECTORY + data_file)
    if lenght_based_padding:
        sampler = BucketBatchSampler(
            dataset.lengths,
            batch_size=batch_size,
            bucket_size=1000,
            shuffle=True
        )
        return torch.utils.data.DataLoader(
            dataset, batch_sampler=sampler,
            collate_fn=lambda batch: pad_collate(batch, tokenizer),
        )
    if weighted_sampler:
        class_weights = torch.tensor([1.2190, 8.6915, 21.0819, 58.2682])
    return torch.utils.data.DataLoader(
        dataset, batch_size=batch_size,
        collate_fn=lambda batch: pad_collate(batch, tokenizer),
    )

def train_and_log(
        run_name: str,
        model: ToxicityClassifier,
        tokenizer: TextTokenizer,
        optimizer: torch.optim.Optimizer,
        criterion: torch.nn.Module,
        train_data_file: str,
        val_data_file: str,
        n_epoch: int = 3,
        batch_size: int = 128,
        lenght_based_padding: bool = False,
        weighted_sampler: bool = False,
        **kwargs
    ):
    train_loader = build_loader(train_data_file, lenght_based_padding, weighted_sampler, batch_size)
    val_loader = build_loader(val_data_file, lenght_based_padding, weighted_sampler, batch_size)

    device = next(model.parameters()).device
    run_params = {
        "model_name": model.__class__.__name__,
        "epochs": n_epoch,
        "batch_size": batch_size,
        "seed": SEED,
        **kwargs
    }

    with mlflow.start_run(run_name=run_name), tqdm(total = len(train_loader) * n_epoch) as pbar:

        mlflow.log_params(run_params)
        mlflow.log_params(get_optimizer_params(optimizer))

        for epoch in range(n_epoch):
            train_loss = train(model, optimizer, criterion, train_loader, device, pbar)
            mlflow.log_metric(
                "train_loss", train_loss, step=epoch
            )

            val_loss, report = validate(model, optimizer, criterion, val_loader, device, pbar)
            mlflow.log_metric(
                "val_loss", val_loss, step=epoch
            )
            class_labels = train_loader.dataset.label_names
            class_ids = train_loader.dataset.label_idx
            for label_idx, label_name in zip(class_ids, class_labels):
                mlflow.log_metric(
                    f"{label_name}-f1-score",
                    report[label_name]["f1-score"],
                    step=epoch
                )
            mlflow.log_metric(
                "macro_f1_score",
                report["macro avg"]["f1-score"],
                step=epoch
            )
        exp = mlflow.get_experiment(mlflow.active_run().info.experiment_id)
        return model.log_model(tokenizer, exp.name)

## BERT_clf_dropout (cointegrated/rubert-tiny2)

In [ ]:
class HFAutoTokenizer(TextTokenizer):

    def __init__(
            self,
            model_name: str,
            padding: bool | str,
            truncation: bool | None = None,
            max_length: int  = None
        ):
        self.tokenizer: AutoTokenizer = AutoTokenizer.from_pretrained(model_name)
        self.padding = padding
        self.truncation = truncation
        self.max_length = max_length

    def encode(self, texts: list[str]):
        output = self.tokenizer(
            texts,
            padding=self.padding,
            truncation=self.truncation,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return output["input_ids"], output["attention_mask"]

    def decode(self, id: int):
        return self.tokenizer.decode(id)

class HFAutoClassifier(ToxicityClassifier):

    def __init__(self, model_name=None, model=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        if model_name is None:
            assert(model is not None)
            self.model = model
            self.model_name = model.config.name_or_path
        else:
            self.model_name = model_name
            self.model = AutoModelForSequenceClassification.from_pretrained(
                model_name,
                num_labels=4
            )

    def forward(self, x, mask):
        return self.model(x, mask).logits

    def log_model(self, tokenizer: HFAutoTokenizer, registered_model_name: str):
        pipe = {
            "tokenizer": tokenizer.tokenizer,
            "model": self.model,
        }
        return mlflow.transformers.log_model(
            transformers_model=pipe,
            name="model",
            task="text-classification",
            registered_model_name=registered_model_name
        )


In [ ]:
model_name = "cointegrated/rubert-tiny2"
mlflow.set_experiment(f"BERT_clf_dropout_{model_name.replace("/", "_")}")

model = HFAutoClassifier(model_name).to(DEVICE)
tokenizer = HFAutoTokenizer(model_name, padding=True, truncation=True, max_length=128)

class_weights = torch.tensor([1.2190, 8.6915, 21.0819, 58.2682], device=DEVICE)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

result = train_and_log(
    "max length (128) padding",
    model,
    tokenizer,
    torch.optim.AdamW(model.parameters(), lr=5e-6),
    loss_fn,
    "train_val.csv",
    "test.csv",
    n_epoch=9,
    batch_size=32,
    class_weights=str(class_weights.tolist())
)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider trai

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026/05/24 21:12:39 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/24 21:12:39 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.25.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.25.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/24 21:12:45 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.25.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.25.0' without the loc

🏃 View run max length (128) padding at: http://70.34.242.179/#/experiments/7/runs/44b491c05c7b4c92ae67cee938b6a737
🧪 View experiment at: http://70.34.242.179/#/experiments/7


## BERT_clf_dropout (DeepPavlov/rubert-base-cased-conversational)

In [ ]:
model_name = "DeepPavlov/rubert-base-cased-conversational"
mlflow.set_experiment(f"BERT_clf_dropout_{model_name.replace("/", "_")}")

model = HFAutoClassifier(model_name).to(DEVICE)
tokenizer = HFAutoTokenizer(model_name, padding=True, truncation=True, max_length=128)

class_weights = torch.tensor([0.1, 0.2, 0.3, 0.4], device=DEVICE)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

result = train_and_log(
    "max length (128) padding + other weights",
    model,
    tokenizer,
    torch.optim.AdamW(model.parameters(), lr=1e-6),
    loss_fn,
    "train.csv",
    "val.csv",
    n_epoch=7,
    batch_size=32,
    class_weights=str(class_weights.tolist())
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased-conversational
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture

Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026/05/25 10:42:45 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/25 10:42:45 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.25.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.25.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/25 10:42:51 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.25.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.25.0' without the loc

🏃 View run max length (128) padding + other weights at: http://70.34.242.179/#/experiments/8/runs/070de11df1b4410887f72b595b3a6df4
🧪 View experiment at: http://70.34.242.179/#/experiments/8


In [ ]:
model_name = "DeepPavlov/rubert-base-cased-conversational"
mlflow.set_experiment(f"BERT_clf_dropout_{model_name.replace("/", "_")}")

model = HFAutoClassifier(model_name).to(DEVICE)
tokenizer = HFAutoTokenizer(model_name, padding=True)

class_weights = torch.tensor([0.1, 0.2, 0.3, 0.4], device=DEVICE)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

result = train_and_log(
    "length based padding + other weights",
    model,
    tokenizer,
    torch.optim.AdamW(model.parameters(), lr=1e-6),
    loss_fn,
    "train_val.csv",
    "test.csv",
    n_epoch=3,
    batch_size=32,
    class_weights=str(class_weights.tolist()),
    lenght_based_padding=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased-conversational
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture

Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026/05/25 11:39:17 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/25 11:39:17 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.25.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.25.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/25 11:39:23 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.25.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.25.0' without the loc

🏃 View run length based padding + other weights at: http://70.34.242.179/#/experiments/8/runs/d041a4da1fac485795b74f489593881d
🧪 View experiment at: http://70.34.242.179/#/experiments/8


## Улучшенная LSTM

In [86]:
class ImprovedLSTM(torch.nn.Module):
    def __init__(self, input_dim=300, hidden_dim=256, output_dim=4):
        super(ImprovedLSTM, self).__init__()
        self.lstm = torch.nn.LSTM(input_dim, hidden_dim, batch_first=True, num_layers=3, dropout=0.4, bidirectional=True)
        self.fc = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim * 2, 128),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.4),
            torch.nn.Linear(128, output_dim)
        )
        self.dropout = torch.nn.Dropout(0.4)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.dropout(out)
        return self.fc(out)

class FasttextTokenizer(TextTokenizer):

    FASTTEXT_MODEL_FILE_NAME = "cc.ru.300.bin"

    def __init__(self):
        fasttext.util.download_model("ru", if_exists="ignore")
        self.fasttext_model = fasttext.load_model(self.FASTTEXT_MODEL_FILE_NAME)


    def encode(self, texts: list[str], max_len=30) -> torch.Tensor:
        embeddings = []
        zero_vector = torch.zeros(300)
        mask = torch.fill_(torch.zeros(max_len), 1)
        for text in texts:
            words = text.split()[:max_len]
            vectors = [
                torch.from_numpy(self.fasttext_model.get_word_vector(word))
                for word in words
            ]

            if len(vectors) < max_len:
                vectors += [zero_vector] * (max_len - len(vectors))

            embeddings.append(torch.vstack(vectors).unsqueeze(0))
        return torch.vstack(embeddings), mask

    def decode(self, id: int) -> str:
        raise NotImplemented()

class LSTMClassifierWrapper(ToxicityClassifier):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.model = ImprovedLSTM()

    def forward(self, x, mask):
        return self.model(x)

    def log_model(self, tokenizer: FasttextTokenizer, registered_model_name: str):
        return mlflow.pytorch.log_model(
            pytorch_model=self.model,
            name="model",
            registered_model_name=registered_model_name,
        )


In [87]:
mlflow.set_experiment("improved_LSTM")

model = LSTMClassifierWrapper().to(DEVICE)
tokenizer = FasttextTokenizer()

loss_fn = torch.nn.CrossEntropyLoss()

result = train_and_log(
    "BiLSTM: 3 layers, hidden_dim=256",
    model,
    tokenizer,
    torch.optim.AdamW(model.parameters(), lr=1e-3),
    loss_fn,
    "train_val.csv",
    "test.csv",
    n_epoch=50,
    batch_size=64,
    weighted_sampler=True
)

100%|█████████▉| 155189/155200 [27:28<00:00, 116.87it/s]2026/05/25 15:07:51 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/05/25 15:07:51 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/25 15:07:54 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this

🏃 View run BiLSTM: 3 layers, hidden_dim=256 at: http://70.34.242.179/#/experiments/11/runs/f6107fccd2f547a6b0fa1d6b3aa82ac7
🧪 View experiment at: http://70.34.242.179/#/experiments/11


## Logistic Regression + BOW

In [78]:
from sklearn.model_selection import train_test_split

LABEL_MAPPING = {
        "NORMAL": 0,
        "INSULT": 1,
        "THREAT": 2,
        "OBSCENITY": 3
}
CLASS_LABELS = ["NORMAL", "INSULT", "THREAT", "OBSCENITY"]
CLASS_IDS = [0, 1, 2, 3]

cleaned_comments = (
    pl.read_csv(DATA_DIRECTORY + "cleaned_comments.csv")
    .with_columns(
        pl.col("label")
        .replace_strict(LABEL_MAPPING)
        .alias("label")
    )
)
X = cleaned_comments["lemmatized_comment"]
y = cleaned_comments["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=cleaned_comments["label"], random_state=SEED)

In [80]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

experiment_name = "baseline_logreg_bow"

mlflow.set_experiment(experiment_name)

with mlflow.start_run(run_name="Logistic Regression + BOW"):
    model = Pipeline([
        ('vectorizer', CountVectorizer()),
        ('logreg', LogisticRegression(max_iter=2000, class_weight='balanced'))
    ]).fit(X_train, y_train)

    y_pred = model.predict(X_test)

    report = classification_report(
        y_test,
        y_pred,
        labels=CLASS_IDS,
        target_names=CLASS_LABELS,
        output_dict=True,
        zero_division=0.0
    )

    for label_idx, label_name in zip(CLASS_IDS, CLASS_LABELS):
        mlflow.log_metric(
            f"{label_name}-f1-score",
            report[label_name]["f1-score"]
        )
        mlflow.log_metric(
            "macro_f1_score",
            report["macro avg"]["f1-score"],
        )

    mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        registered_model_name=experiment_name
    )

2026/05/25 14:05:48 INFO mlflow.tracking.fluent: Experiment with name 'baseline_logreg_bow' does not exist. Creating a new experiment.
2026/05/25 14:06:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'baseline_logreg_bow'.
2026/05/25 14:06:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: baseline_logreg_bow, version 1
Created version '1' of model 'baseline_logreg_bow'.


🏃 View run Logistic Regression + BOW at: http://70.34.242.179/#/experiments/10/runs/3093f40ebe70453fa06e5fc5ef6c5c17
🧪 View experiment at: http://70.34.242.179/#/experiments/10
